# Tutorial 11 - Predictive versus generative modelling

By the end of this section, students will be able to:

1. Give examples of questions that can be answered by generative models and others that can be answered by predictive models.
2. Discuss how the research question being asked impacts the statistical modelling procedures.
3. Discuss why the model obtained directly from lasso is not the most suitable model for generative modelling and how post-lasso is one way to address this problem.
4. Write a computer script to perform post-lasso and use it to estimate a generative model.
5. Discuss post inference problems (e.g., double dipping into the data set) and current practical solutions available to address these (e.g., data-splitting techniques).
6. Write a computer script to apply currently available practical solutions to post inference problems.
7. Discuss how the research question being asked impacts the communication of the results.

In [ ]:
# Loading packages
library(car)
library(tidyverse)
library(broom)
library(glmnet)
library(leaps)
source("tests_tutorial_11.r")

## Model Selection For Inference

In this tutorial, you will go through the model selection process in a real data set. Remember the [Ames `Housing` dataset](https://www.kaggle.com/c/home-data-for-ml-course/) you worked on Worksheet 08? Let's refresh our memory: it was compiled by Dean De Cock, it has 79 input variables on different characteristics of residential houses in Ames, Iowa, USA that can be used to predict the property's final price, `SalePrice`. As in worksheet_08, we will focus our attention on 21 numerical input variables:

- `LotFrontage`: Linear $\text{ft}$ of street connected to the house.
- `LotArea`: Lot size in $\text{ft}^2$.
- `MasVnrArea`: Masonry veneer area in $\text{ft}^2$.
- `TotalBsmtSF`: Total $\text{ft}^2$ of basement area.
- `GrLivArea`: Above grade (ground) living area in $\text{ft}^2$.
- `BsmtFullBath`: Number of full bathrooms in basement.
- `BsmtHalfBath`: Number of half bathrooms in basement.
- `FullBath`: Number of full bathrooms above grade.
- `HalfBath`: Number of half bathroom above grade.
- `BedroomAbvGr`: Number of bedrooms above grade (it does not include basement bedrooms).
- `KitchenAbvGr`: Number of kitchens above grade.
- `Fireplaces`: Number of fireplaces.
- `GarageArea`: Garage's area in $\text{ft}^2$.
- `WoodDeckSF`: Wood deck area in $\text{ft}^2$.
- `OpenPorchSF`: Open porch area in $\text{ft}^2$.
- `EnclosedPorch`: Enclosed porch area in $\text{ft}^2$.
- `ScreenPorch`: Screen porch area in $\text{ft}^2$.

Let's start by loading the data set. 

In [ ]:
## Load the housing data set
housing_raw <- read_csv("data/Housing.csv", col_types = cols())

# Use `YearBuilt` and `YrSold` to create a variable `ageSold`
housing_raw$ageSold <- housing_raw$YrSold - housing_raw$YearBuilt


# Select subset of input variables
housing_raw <- 
  housing_raw %>%
  select(Id,
    LotFrontage, LotArea, MasVnrArea, TotalBsmtSF, 
    GrLivArea, BsmtFullBath, BsmtHalfBath, FullBath, HalfBath, BedroomAbvGr, KitchenAbvGr, Fireplaces,
    GarageArea, WoodDeckSF, OpenPorchSF, EnclosedPorch, ScreenPorch, PoolArea, ageSold, SalePrice
  )

# Remove those rows containing `NA`s and some outliers
housing_raw <- 
    drop_na(housing_raw)  %>% 
    filter(LotArea < 20000)

str( housing_raw )

Our objective in this tutorial is to obtain a model for inference. We want to study how the properties' values are affected by the different properties' attributes. We want to be able to:

1. Interpret the parameters of the model;
2. Identify relevant attributes (covariates); 
3. Have a measure of uncertainty of our estimates.

**Question 1.1** 
<br> {points: 1}

Since we do not know which variables are important/relevant, we will need to conduct a variable selection technique. Let's start by splitting the data set into two sets: (1) the first part, with 70% of the rows, will be for inference; and (2) the second part, will be for variable selection. 

Your job is to randomly select 70% of the rows and store them in an object called `housing_inference`. Store the remaining rows in an object called `housing_selection`.

The `housing_inference` object is golden! It should not be touched before we select the variables. No peeking!!  (Hint: you might want to check the [slice_sample](https://dplyr.tidyverse.org/reference/slice.html) and [anti_join](https://dplyr.tidyverse.org/articles/base.html?q=anti%20joi#filtering-joins) functions.)

In [ ]:
set.seed(20211118) # Do not change this

# housing_inference <- 
#     ... %>% 
#     slice_sample(... = 0.7)

# housing_selection <- 
#     housing_raw %>% 
#     anti_join(...)


# your code here
fail() # No Answer - remove if you provide an answer

head(housing_selection)

In [ ]:
test_1.1()

Good work! Now let's remove the `Id` column from both datasets.

In [ ]:
housing_selection <-
    housing_selection %>% 
    select(-Id)

housing_inference <-
    housing_inference %>% 
    select(-Id)

**Question 1.2** 
<br> {points: 1}

As we discussed in the worksheet, there are many possible approaches for model selection. Let's focus on Lasso. Run Lasso on the `housing_selection` tibble and find the value `lambda` that provides the lowest Cross-validation MSE. (See `cv.glmnet` function.)

_Save the result in an object named `lasso_model`._

In [ ]:
set.seed(20211118) # do not change this

# lasso_model <-
#     cv.glmnet(... %>% as.matrix(), 
#               ..., 
#               alpha = ...)

# your code here
fail() # No Answer - remove if you provide an answer

lasso_model

In [ ]:
test_1.2()

**Question 1.3** 
<br> {points: 1}

Obtain the coefficients of the best lasso model found in the `lasso_model`. By best, we mean the one with the smallest MSE. 

_Save the result in an object named `beta_lasso`._

In [ ]:
set.seed(20211118) # do not change this


# your code here
fail() # No Answer - remove if you provide an answer

beta_lasso

In [ ]:
test_1.3()

**Question 1.4** 
<br> {points: 1}

Next, we shall save the covariates selected by our Lasso model in an object named `lasso_selected_covariates`.  

In [ ]:
#lasso_selected_covariates <-

# your code here
fail() # No Answer - remove if you provide an answer

lasso_selected_covariates

In [ ]:
test_1.4()

**Question 1.5** 
<br> {points: 1}

We expect that Lasso would remove highly correlated variables. However, Lasso can still fit a linear model on data sets with high levels of multicollinearity. Unfortunately, ordinary least squares cannot. To be on the safe side, let's check the variance inflator factor of the variables selected by Lasso. 

_Save the output in an object named `lasso_variables_vif`._

In [ ]:
#lasso_variables_vif <- 
#    vif(...)

# your code here
fail() # No Answer - remove if you provide an answer

lasso_variables_vif

In [ ]:
test_1.5()

**Question 1.6**
<br>{points: 1}

True or false?

The `lasso_variables_vif` does not indicate a very concerning presence of multicollinearity. 

_Assign your answer to an object called `answer1.6`. Your answer should be either "true" or "false", surrounded by quotes._

In [ ]:
# answer1.6 <- ...

# your code here
fail() # No Answer - remove if you provide an answer

In [ ]:
test_1.6()

**Question 1.7** 
<br> {points: 1}

Finally, let's use the covariates selected by lasso and stored in `lasso_selected_covariates` to fit a linear model using ordinary least squares.

_Save the output in an object named `inference_model`._

In [ ]:
# your code here
fail() # No Answer - remove if you provide an answer

summary(inference_model)

In [ ]:
test_1.7()

**Question 1.8** 
<br> {points: 1}

The model stored in `inference_model` has shown 3 non-significant variables. Should we remove these variables and re-fit the model with them? Briefly explain why or why not. 

DOUBLE CLICK TO EDIT **THIS CELL** AND REPLACE THIS TEXT WITH YOUR ANSWER.